In [1]:
from lexico import *
from sintactico_ast import *
import json

In [2]:
codigo_fuente = """
int suma(int a, int b) {
    int c = a + b;
    return c;
}

int main() {
    int resultado = suma(3, 5);
    return 0;
}
"""

print("--- 1. CÓDIGO FUENTE ---")
print(codigo_fuente)

print("\n--- 2. TOKENS GENERADOS (Léxico) ---")
tokens = identificar_tokens(codigo_fuente)
parser = Parser(tokens)
arbol = parser.parsear()
for token in tokens:
    print(token)

print("\n[Traducción a GO]")
print("package main")
print("import \"fmt\"")
print(arbol.traducirGo())

--- 1. CÓDIGO FUENTE ---

int suma(int a, int b) {
    int c = a + b;
    return c;
}

int main() {
    int resultado = suma(3, 5);
    return 0;
}


--- 2. TOKENS GENERADOS (Léxico) ---
('KEYWORD', 'int')
('IDENTIFIER', 'suma')
('DELIMITER', '(')
('KEYWORD', 'int')
('IDENTIFIER', 'a')
('DELIMITER', ',')
('KEYWORD', 'int')
('IDENTIFIER', 'b')
('DELIMITER', ')')
('DELIMITER', '{')
('KEYWORD', 'int')
('IDENTIFIER', 'c')
('OPERATOR', '=')
('IDENTIFIER', 'a')
('OPERATOR', '+')
('IDENTIFIER', 'b')
('DELIMITER', ';')
('KEYWORD', 'return')
('IDENTIFIER', 'c')
('DELIMITER', ';')
('DELIMITER', '}')
('KEYWORD', 'int')
('IDENTIFIER', 'main')
('DELIMITER', '(')
('DELIMITER', ')')
('DELIMITER', '{')
('KEYWORD', 'int')
('IDENTIFIER', 'resultado')
('OPERATOR', '=')
('IDENTIFIER', 'suma')
('DELIMITER', '(')
('NUMBER', '3')
('DELIMITER', ',')
('NUMBER', '5')
('DELIMITER', ')')
('DELIMITER', ';')
('KEYWORD', 'return')
('NUMBER', '0')
('DELIMITER', ';')
('DELIMITER', '}')

[Traducción a GO]
package main


In [3]:
import json

import json

def imprimir_ast(nodo):
    # 0. Nodo Raíz (Programa)
    if isinstance(nodo, NodoPrograma):
        return {
            "Tipo": "Programa",
            "Funciones": [imprimir_ast(f) for f in nodo.funcion]
        }
        
    # 1. Nodos Principales
    elif isinstance(nodo, NodoFuncion):
        return {
            "Tipo": "Funcion",
            "Nombre": nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre,
            "Retorno": nodo.tipo[1] if isinstance(nodo.tipo, tuple) else nodo.tipo,
            "Parametros": [imprimir_ast(p) for p in nodo.parametros],
            "Cuerpo": [imprimir_ast(c) for c in nodo.cuerpo]
        }
    
    # 2. Llamadas y Prints
    elif isinstance(nodo, NodoLlamada):
        return {
            "Tipo": "LlamadaFuncion",
            "Nombre": nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre,
            "Argumentos": [imprimir_ast(a) for a in nodo.argumentos]
        }
    elif isinstance(nodo, NodoPrint):
        return {
            "Tipo": "SentenciaPrint",
            "Expresiones": [imprimir_ast(e) for e in nodo.expresiones]
        }
    elif isinstance(nodo, NodoString):
        return {
            "Tipo": "String",
            "Valor": nodo.valor[1] if isinstance(nodo.valor, tuple) else nodo.valor
        }

    # 3. Nodos Base
    elif isinstance(nodo, NodoParametros):
        return {
            "Tipo": "Parametro",
            "Nombre": nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre,
            "Dato": nodo.tipo[1] if isinstance(nodo.tipo, tuple) else nodo.tipo
        }
    elif isinstance(nodo, NodoAsignacion):
        return {
            "Tipo": "Asignacion",
            "Variable": nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre,
            "Expresion": imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoOperacion):
        return {
            "Tipo": "Operacion",
            "Operador": nodo.operador[1] if isinstance(nodo.operador, tuple) else nodo.operador,
            "Izquierda": imprimir_ast(nodo.izquierda),
            "Derecha": imprimir_ast(nodo.derecha)
        }
    elif isinstance(nodo, NodoRetorno):
        return {
            "Tipo": "Retorno",
            "Valor": imprimir_ast(nodo.expresion)
        }
    elif isinstance(nodo, NodoNumero):
        return {"Numero": nodo.valor[1] if isinstance(nodo.valor, tuple) else nodo.valor}
    elif isinstance(nodo, NodoIdentificador):
        return {"Identificador": nodo.nombre[1] if isinstance(nodo.nombre, tuple) else nodo.nombre}
    
    return {"Error": "Nodo desconocido"}

In [4]:
# 1. Tokenizar y Parsear
tokens = identificar_tokens(codigo_fuente)
parser = Parser(tokens)
arbol = parser.parsear()

# 2. Convertir a JSON
ast_json = json.dumps(imprimir_ast(arbol), indent=2)

# 3. Imprimir para copiar
print(ast_json)

{
  "Tipo": "Programa",
  "Funciones": [
    {
      "Tipo": "Funcion",
      "Nombre": "suma",
      "Retorno": "int",
      "Parametros": [
        {
          "Tipo": "Parametro",
          "Nombre": "a",
          "Dato": "int"
        },
        {
          "Tipo": "Parametro",
          "Nombre": "b",
          "Dato": "int"
        }
      ],
      "Cuerpo": [
        {
          "Tipo": "Asignacion",
          "Variable": "c",
          "Expresion": {
            "Tipo": "Operacion",
            "Operador": "+",
            "Izquierda": {
              "Identificador": "a"
            },
            "Derecha": {
              "Identificador": "b"
            }
          }
        },
        {
          "Tipo": "Retorno",
          "Valor": {
            "Identificador": "c"
          }
        }
      ]
    },
    {
      "Tipo": "Funcion",
      "Nombre": "main",
      "Retorno": "int",
      "Parametros": [],
      "Cuerpo": [
        {
          "Tipo": "Asignacion",
       